In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/Sem Redução de Dimensionalidade/'

nome_dados_treinamento = 'lycos_cicids2017_treinamento_sem_XSS.csv'
nome_dados_teste = 'lycos_cicids2017_teste_com_XSS.csv'

In [3]:
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")
display(df_treino.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1285780, 80) | Teste: (551718, 80)


,src_port,dst_port,ip_prot,timestamp,flow_duration,down_up_ratio,pkt_len_max,pkt_len_min,pkt_len_mean,pkt_len_var,...,bwd_bulk_bytes_mean,bwd_bulk_pkt_mean,bwd_bulk_rate_mean,fwd_subflow_bytes_mean,fwd_subflow_pkt_mean,bwd_subflow_bytes_mean,bwd_subflow_pkt_mean,fwd_tcp_init_win_bytes,bwd_tcp_init_win_bytes,Label
0,0.951110,0.000809,0.122137,0.493151,0.000504,0.123909,0.003948,0.031768,0.030108,0.000050,...,0.000000,0.000000,0.000000,0.000026,0.000005,1.495151e-07,0.000003,0.000000,0.000000,benign
1,0.766598,0.006760,0.038168,0.235429,0.025492,0.041303,0.001249,0.000000,0.003345,0.000009,...,0.000000,0.000000,0.000000,0.000009,0.000007,0.000000e+00,0.000002,0.003922,0.021423,benign
2,0.814466,0.006760,0.038168,0.512447,0.979619,0.111518,0.057131,0.000000,0.066731,0.004397,...,0.000004,0.000069,0.000009,0.000228,0.000030,2.463439e-06,0.000021,0.445572,0.647110,benign
3,0.074983,0.000809,0.122137,0.240826,0.000002,0.123909,0.010475,0.022790,0.061262,0.000639,...,0.000000,0.000000,0.000000,0.000037,0.000009,7.933453e-07,0.000007,0.000000,0.000000,benign
4,0.847898,0.006760,0.038168,0.235855,0.070290,0.149890,0.175020,0.000000,0.392759,0.041693,...,0.000387,0.000566,0.000418,0.000322,0.000094,6.456254e-05,0.000086,0.445572,0.176773,benign


In [4]:
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

In [5]:
print("Codificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
num_classes = len(encoder.classes_)

Codificando as labels para a Rede Neural...


In [6]:
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [7]:
print("\nIniciando o treinamento da Rede Neural Profunda (DNN)...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN)...
Epoch 1/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9882 - loss: 0.0522 - val_accuracy: 0.9972 - val_loss: 0.0100
Epoch 2/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9970 - loss: 0.0103 - val_accuracy: 0.9980 - val_loss: 0.0065
Epoch 3/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9979 - loss: 0.0070 - val_accuracy: 0.9987 - val_loss: 0.0050
Epoch 4/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9982 - loss: 0.0059 - val_accuracy: 0.9987 - val_loss: 0.0047
Epoch 5/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9983 - loss: 0.0052 - val_accuracy: 0.9988 - val_loss: 0.0040
Epoch 6/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9985 - loss: 0.0047 - val_accuracy: 0.9988 - val_loss: 0.0038
Epoch 7/10
4521/4521 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9985 - loss: 0.0045 - val_accuracy: 0.9988 - val_loss: 0.0035
Epoch 8/10
4521/4521 ━━━━━━━━━━━

In [8]:
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

print("\n=== MATRIZ DE CONFUSÃO (DNN — Sem XSS no treino) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
17242/17242 ━━━━━━━━━━━━━━━━━━━━ 8s 457us/step
Classificação concluída! Tempo de previsão: 12.1999 segundos.

=== MATRIZ DE CONFUSÃO (DNN — Sem XSS no treino) ===


,Pred_benign,Pred_bot,Pred_ddos,Pred_dos_goldeneye,Pred_dos_hulk,Pred_dos_slowhttptest,Pred_dos_slowloris,Pred_ftp_patator,Pred_heartbleed,Pred_portscan,Pred_ssh_patator,Pred_webattack_bruteforce,Pred_webattack_sql_injection,Pred_webattack_xss
Real_benign,418195,1,1,205,1,84,10,0,0,20,1,121,0,0
Real_bot,1,218,0,0,0,0,0,0,0,0,0,0,0,0
Real_ddos,1,0,28587,0,0,0,0,0,0,8,0,0,0,0
Real_dos_goldeneye,56,0,0,2038,1,1,0,0,0,0,0,0,0,0
Real_dos_hulk,2,0,0,0,47934,3,3,0,0,0,0,0,0,0
Real_dos_slowhttptest,15,0,0,1,0,1404,3,0,0,0,0,0,0,0
Real_dos_slowloris,5,0,0,0,0,10,1760,0,0,0,0,0,0,0
Real_ftp_patator,3,0,0,0,0,0,0,1178,0,0,0,0,0,0
Real_heartbleed,1,0,0,0,0,0,0,0,3,0,0,0,0,0
Real_portscan,5,0,0,0,0,0,0,0,0,47902,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                         precision    recall  f1-score   support

                 benign       1.00      1.00      1.00    418639
                    bot       1.00      1.00      1.00       219
                   ddos       1.00      1.00      1.00     28596
          dos_goldeneye       0.91      0.97      0.94      2096
               dos_hulk       1.00      1.00      1.00     47942
       dos_slowhttptest       0.93      0.99      0.96      1423
          dos_slowloris       0.99      0.99      0.99      1775
            ftp_patator       1.00      1.00      1.00      1181
             heartbleed       1.00      0.75      0.86         4
               portscan       1.00      1.00      1.00     47907
            ssh_patator       1.00      0.99      0.99       875
   webattack_bruteforce       0.34      0.97      0.50       397
webattack_sql_injection       0.00      0.00      0.00         3
          webattack_xss      

In [9]:
CLASS_ALIASES_LATEX = {'benign': 'BENIGN', 'bot': 'Bot', 'ddos': 'DDoS', 'dos_goldeneye': 'DoS GE', 'dos_hulk': 'Hulk', 'dos_slowhttptest': 'Slowhttp', 'dos_slowloris': 'Slowloris', 'ftp_patator': 'FTP', 'heartbleed': 'Heart', 'portscan': 'PortScan', 'ssh_patator': 'SSH', 'webattack_bruteforce': 'Brute', 'webattack_sql_injection': 'SQL', 'webattack_xss': 'XSS'}


def escape_latex(value):
    replacements = {
        "\\": "\\textbackslash{}",
        "&": "\\&",
        "%": "\\%",
        "$": "\\$",
        "#": "\\#",
        "_": "\\_",
        "{": "\\{",
        "}": "\\}",
        "~": "\\textasciitilde{}",
        "^": "\\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return f"\\ok{{{value}}}"
    if value != 0:
        return f"\\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((f"Real\\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\scriptsize",
        "    \\resizebox{\\linewidth}{!}{",
        f"        \\begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        "            \\hline",
        format_row("", headers),
        "            \\hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        "            \\hline",
        "        \\end{tabular}",
        "    }",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        ["\\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        ["\\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        ["\\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + " \\\\"

    lines = [
        "\\begin{table}[H]",
        "    \\centering",
        "    \\small",
        "    \\begin{tabular}{lrrrr}",
        "        \\hline",
        format_row(headers),
        "        \\hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        "        \\hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        "        \\hline",
        "    \\end{tabular}",
        f"    \\caption{{{escape_latex(caption)}}}",
        f"    \\label{{{table_label}}}",
        "\\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_mc = make_latex_confusion_matrix(
    cm, labels_latex,
    "Matriz de Confusão — DNN (LycoS-IDS2017, Sem XSS no treino)",
    "table:dnn_lycos_sem_xss_mc",
)
latex_metricas = make_latex_metrics_report(
    y_true_latex, y_pred_latex, labels_latex,
    "Relatório de Métricas — DNN (LycoS-IDS2017, Sem XSS no treino)",
    "table:dnn_lycos_sem_xss_metricas",
)

print(latex_mc)
print("\n")
print(latex_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrr}
            \hline
                            & BENIGN      & Bot      & DDoS       & DoS GE    & Hulk       & Slowhttp  & Slowloris & FTP       & Heart  & PortScan   & SSH      & Brute     & SQL    & XSS    \\
            \hline
            Real\_BENIGN    & \ok{418195} & \err{1}  & \err{1}    & \err{205} & \err{1}    & \err{84}  & \err{10}  & 0         & 0      & \err{20}   & \err{1}  & \err{121} & 0      & 0      \\
            Real\_Bot       & \err{1}     & \ok{218} & 0          & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0         & 0      & 0      \\
            Real\_DDoS      & \err{1}     & 0        & \ok{28587} & 0         & 0          & 0         & 0         & 0         & 0      & \err{8}    & 0        & 0         & 0      & 0      \\
            Real\_DoS GE    & \err{56}    & 0        & 0          & \